# NB05 – Da amostra à decisão
**Métodos e Ferramentas de Data Science – FGV MBA · Aula 5**

## A pergunta de negócio
Na Aula 3, vimos que pedidos atrasados têm nota média menor. A diretoria agora pergunta:

> **Essa diferença é real ou pode ser acaso? Se olharmos só uma amostra de pedidos, quanto podemos confiar no número?**

## A ideia desta aula
Vamos tratar a base completa da Olist como se fosse a **população** (todos os pedidos) e tirar **amostras** dela. Assim podemos ver, na prática, o que acontece quando decidimos com uma amostra, que é o que quase sempre fazemos (pesquisas, pilotos, testes A/B, auditorias).

## 1. Montar a população

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

BASE = "https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/"
pedidos = pd.read_csv(BASE + "olist_pedidos.csv.gz",
                      parse_dates=["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"])
avaliacoes = pd.read_csv(BASE + "olist_avaliacoes.csv.gz")

pop = pedidos[(pedidos["order_status"] == "delivered") & pedidos["order_delivered_customer_date"].notna()].copy()
pop["atrasou"] = pop["order_delivered_customer_date"] > pop["order_estimated_delivery_date"]
pop["dias_entrega"] = (pop["order_delivered_customer_date"] - pop["order_purchase_timestamp"]).dt.days
pop = pop.merge(avaliacoes.groupby("order_id")["review_score"].mean().rename("nota"), on="order_id").dropna(subset=["nota"])

MEDIA_VERDADEIRA = pop["nota"].mean()
print(f"População: {len(pop):,} pedidos | nota média verdadeira: {MEDIA_VERDADEIRA:.3f}".replace(",", "."))

## 2. Uma amostra erra. Quanto?
Tiramos 100 pedidos ao acaso e calculamos a nota média. Depois repetimos isso 1.000 vezes.

In [ ]:
rng = np.random.default_rng(42)
uma = pop["nota"].sample(100, random_state=1)
print(f"Uma amostra de 100 pedidos: nota média {uma.mean():.2f} (verdade: {MEDIA_VERDADEIRA:.2f})")

medias_100 = [pop["nota"].sample(100, random_state=int(s)).mean() for s in rng.integers(0, 1_000_000, 1000)]
medias_1000 = [pop["nota"].sample(1000, random_state=int(s)).mean() for s in rng.integers(0, 1_000_000, 1000)]

plt.figure(figsize=(9, 4))
sns.histplot(medias_100, bins=40, stat="density", label="amostras de 100", alpha=0.5)
sns.histplot(medias_1000, bins=40, stat="density", label="amostras de 1.000", alpha=0.5, color="darkorange")
plt.axvline(MEDIA_VERDADEIRA, color="black", linestyle="--", label="média verdadeira")
plt.title("Médias de 1.000 amostras: quanto maior a amostra, menor o erro")
plt.xlabel("nota média da amostra"); plt.legend(); plt.show()
print(f"Desvio das médias: n=100 → {np.std(medias_100):.3f} | n=1.000 → {np.std(medias_1000):.3f}")

**Como ler:** as médias das amostras formam uma curva em forma de sino em torno da verdade (é o **teorema central do limite**).
Com 10 vezes mais pedidos, o erro típico cai para cerca de um terço (√10 ≈ 3,2). **Precisão custa caro:** dobrar a precisão exige quatro vezes mais dados.

## 3. Intervalo de confiança: o número com sua incerteza
De **uma** amostra, estimamos um intervalo que provavelmente contém a verdade:

> média ± 1,96 × (desvio padrão ÷ √n)

"95% de confiança" quer dizer: se repetíssemos o processo muitas vezes, cerca de 95% dos intervalos conteriam a média verdadeira.

In [ ]:
def intervalo(amostra, conf=0.95):
    m, ep = amostra.mean(), amostra.std(ddof=1) / np.sqrt(len(amostra))   # ep = erro padrão
    lo, hi = stats.t.interval(conf, len(amostra) - 1, loc=m, scale=ep)
    return m, lo, hi

m, lo, hi = intervalo(uma)
print(f"Amostra de 100: {m:.2f}, IC 95% [{lo:.2f}; {hi:.2f}]")

# Quantos de 100 intervalos contêm a verdade?
acertos = 0
for s in range(100):
    _, lo, hi = intervalo(pop["nota"].sample(100, random_state=1000 + s))
    acertos += lo <= MEDIA_VERDADEIRA <= hi
print(f"{acertos} de 100 intervalos contêm a média verdadeira")

## 4. Teste de hipótese: a diferença é real ou acaso?
**Hipótese nula:** atrasados e não atrasados têm a mesma nota média; a diferença na amostra é só acaso.
O **p-valor** é a probabilidade de ver uma diferença tão grande quanto a observada **se a hipótese nula fosse verdadeira**.

**O p-valor NÃO é:** a probabilidade de a hipótese ser verdadeira, nem o tamanho ou a importância do efeito.

In [ ]:
amostra = pop.sample(300, random_state=7)
atrasados = amostra.loc[amostra["atrasou"], "nota"]
no_prazo = amostra.loc[~amostra["atrasou"], "nota"]

t, p = stats.ttest_ind(atrasados, no_prazo, equal_var=False)   # teste t de Welch
dif = atrasados.mean() - no_prazo.mean()
print(f"Amostra de 300 pedidos: {len(atrasados)} atrasados e {len(no_prazo)} no prazo")
print(f"Diferença de nota: {dif:.2f} pontos | p-valor: {p:.2g}")

Mesmo com só 300 pedidos (e poucos atrasados), a diferença é grande demais para ser acaso: p-valor muito pequeno.
Agora o contrário: uma diferença **pequena** com uma amostra **gigante**.

## 5. Significativo não quer dizer importante

In [ ]:
pop["mes_par"] = pop["order_purchase_timestamp"].dt.month % 2 == 0
a, b = pop.loc[pop["mes_par"], "nota"], pop.loc[~pop["mes_par"], "nota"]
t, p = stats.ttest_ind(a, b, equal_var=False)
print(f"Meses pares × ímpares: diferença de {a.mean() - b.mean():.3f} pontos | p-valor {p:.2g}")

Com quase 100 mil pedidos, até diferenças minúsculas podem dar "significativas". A pergunta da diretoria não é "é significativo?", e sim **"o efeito é grande o bastante para mudar a decisão?"**.
Sempre reporte o **tamanho do efeito** com seu **intervalo**, não só o p-valor.

## 6. Correlação não é causalidade (de novo)
Por estado, o prazo médio de entrega e a nota média andam juntos. Isso prova que o prazo derruba a nota?

In [ ]:
clientes = pd.read_csv(BASE + "olist_clientes.csv.gz")
uf = (pop.merge(clientes[["customer_id", "customer_state"]], on="customer_id")
         .groupby("customer_state").agg(prazo=("dias_entrega", "mean"), nota=("nota", "mean"), pedidos=("order_id", "count")))
r, p = stats.pearsonr(uf["prazo"], uf["nota"])
print(f"Correlação entre estados: r = {r:.2f} (p = {p:.2g})")
sns.scatterplot(data=uf, x="prazo", y="nota", size="pedidos", legend=False)
for s, row in uf.iterrows():
    plt.annotate(s, (row["prazo"], row["nota"]), fontsize=8)
plt.title("Estados: prazo médio × nota média"); plt.xlabel("prazo médio (dias)"); plt.ylabel("nota média"); plt.show()

A correlação é moderada e negativa (r ≈ −0,45, com só 27 estados), mas outras coisas variam junto com o estado: renda, tipo de produto comprado, frete, expectativa do cliente.
Para afirmar causa, precisaríamos de um **experimento** (por exemplo, prometer prazos diferentes a grupos sorteados) ou de uma análise que controle essas variáveis.

## 7. Do número à decisão: storytelling com dados
Um bom gráfico para a diretoria tem **um título que é a mensagem**, não a descrição dos eixos, e mostra a **incerteza**.

In [ ]:
resumo = pop.groupby("atrasou")["nota"].agg(["mean", "std", "count"])
resumo["ep"] = resumo["std"] / np.sqrt(resumo["count"])
resumo.index = ["No prazo", "Atrasado"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(resumo.index, resumo["mean"], yerr=1.96 * resumo["ep"], color=["#9bb7cc", "#b4443a"], capsize=6)
for i, v in enumerate(resumo["mean"]):
    ax.text(i, v + 0.08, f"{v:.1f}", ha="center", fontsize=12, fontweight="bold")
ax.set_ylim(0, 5)
ax.set_title("Atraso derruba a nota de 4,3 para 2,6", fontsize=13, loc="left")
ax.set_ylabel("nota média (1 a 5)")
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)
plt.show()

### Estrutura de uma mensagem executiva
1. **Achado** (o título): "Atraso derruba a nota de 4,3 para 2,6."
2. **Evidência:** número, comparação e tamanho do efeito.
3. **Incerteza e limitações:** intervalo, amostra, o que não sabemos (causa, período).
4. **Recomendação:** o que fazer, com uma condição verificável ("priorizar as rotas do Nordeste e medir a nota em 3 meses").

---
## 8. Sua vez: exercícios individuais

**Exercício 1.** Tire uma amostra de 200 pedidos (`random_state=7`) e calcule o **intervalo de confiança de 95%** do prazo médio de entrega (`dias_entrega`). O intervalo contém o prazo médio verdadeiro da população?

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 2.** Repita o exercício 1 com 2.000 pedidos. Quanto o intervalo encolheu?

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 3.** Na amostra de 300 pedidos da seção 4, teste se o **prazo de entrega** (`dias_entrega`) dos atrasados é diferente do dos entregues no prazo. Qual o p-valor? A diferença é relevante para o cliente?

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 4.** Reescreva o título do gráfico abaixo como uma **mensagem**:
> "Nota média por status de entrega"

E escreva, em 3 frases, a mensagem executiva completa (achado, incerteza, recomendação).

In [ ]:
titulo = ""     # SEU CÓDIGO AQUI
mensagem = ""   # SEU CÓDIGO AQUI
print(titulo); print(mensagem)

---
## Erros comuns

| Mensagem | O que aconteceu | O que fazer |
|---|---|---|
| `NameError: name 'stats' is not defined` | A primeira célula não foi executada | Execute a seção 1 |
| `ValueError: a must be greater than 0` ou amostra vazia | Filtro sem nenhuma linha | Confira a condição do filtro |
| Os números mudam a cada execução | Amostra sem `random_state` | Fixe `random_state` para reproduzir |
| p-valor aparece como `0.0` | Valor menor que a precisão do computador | Reporte como "p < 0,001" |

## Plano B
Se o download falhar, carregue `olist_pedidos.csv.gz`, `olist_avaliacoes.csv.gz` e `olist_clientes.csv.gz` pelo painel de arquivos do Colab e troque `BASE = ...` por `BASE = ""`.